# GPT-4o 到 GPT-5.1 迁移指南

---

## ⚠️ GPT-4o 退役计划

Azure OpenAI 将于 **2026 年 3 月 31 日** 对以下 GPT-4o 版本执行退役操作：

| 模型版本 | 退役日期 | 替代模型 |
|----------|----------|----------|
| `gpt-4o` (2024-05-13) | 2026-03-31 | `gpt-5.1` (2025-11-13) |
| `gpt-4o` (2024-08-06) | 2026-03-31 | `gpt-5.1` (2025-11-13) |

> 📖 参考：[模型退役时间表](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/model-retirements?view=foundry-classic&tabs=text#text-generation-1) | [模型版本工作原理](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/model-versions?view=foundry-classic#how-model-versions-work)

### Azure 会自动执行的操作：
1. **自动升级**：在退役日期，Azure 会自动将现有 GPT-4o 部署升级到 GPT-5.1
2. **部署名称保持不变**：您的部署名称不会改变，但底层模型会变更
3. **API 端点不变**：您的 API 端点 URL 保持不变

> ⛔ **重要警告**：如果您的部署设置了 **"No Auto Upgrade"（不自动升级）**，该部署将在模型退役后 **停止工作**！请确保在退役日期前手动升级或更改升级策略。
> 
> 📖 参考：[模型版本工作原理](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/model-versions?view=foundry-classic#how-model-versions-work)

### ⚠️ 为什么需要提前迁移？
虽然 Azure 会自动升级，但由于 **GPT-4o 和 GPT-5.1 的 API 参数存在差异**，如果不修改代码：
- 使用 `max_tokens` 参数的代码 **会直接报错 (400 Bad Request)**
- 无法利用 GPT-5.1 的推理能力

此外，**两个模型的性能和行为特性存在差异**，这意味着：
- 相同的 System Prompt 在两个模型上的表现可能不一致
- 输出格式、语气、详细程度等可能有所不同
- 对某些指令的理解和执行方式可能存在差异

**建议**：在退役日期前主动测试并迁移代码，特别是要 **验证您的终端应用所使用的 System Prompt 在 GPT-5.1 上的表现是否符合预期**，确保平稳过渡。

### ❓ 关于区域可用性的常见问题

**问**：目前 GPT-4o (2024-08-06) 在几乎所有区域都可用，但 GPT-5.1 只在少数区域可用。如果我的部署在一个 GPT-5.1 尚未上线的区域，会发生什么？

**答**：根据 [官方文档](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/model-versions?view=foundry-classic#will-a-model-upgrade-happen-if-the-new-model-version-is-not-yet-available-in-that-region)，**即使新模型版本尚未在该区域可用，Azure 仍会在计划的升级窗口期间自动升级部署**。Azure 工程团队会从宣布的升级日期开始在相关区域部署新模型版本，以完成默认模型版本升级流程。

> 📖 参考：[模型区域可用性](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?view=foundry-classic&tabs=global-standard-aoai%2Cglobal-standard&pivots=azure-openai#global-standard-model-availability)

---
## 关键差异总结

| 特性 | GPT-4o (2024-08-06) | GPT-5.1 (2025-11-13) |
|------|---------------------|----------------------|
| **上下文窗口** | 128K tokens | 400K tokens |
| **最大输出** | 16,384 tokens | 128,000 tokens |
| **推理能力** | ❌ | ✅ (默认 `none`) |
| **Chat Completions 输出限制参数** | `max_tokens` | `max_completion_tokens` |
| **Responses API 输出限制参数** | `max_output_tokens` | `max_output_tokens` |
| **`reasoning_effort`** | ❌ 不支持 | ✅ 支持 (`none`, `minimal`, `low`, `medium`, `high`) |
| **推理模式下 temperature** | N/A | ❌ 不支持（会报错） |


---



本 Notebook 演示使用 **REST API** (而非 OpenAI SDK) 从 `gpt-4o` 迁移到 `gpt-5.1` 的完整过程。

## 涵盖内容

### Part 1: Chat Completions API
1. 切换前：GPT-4o 的标准 REST API 调用
2. 切换后（不修改代码）：直接换模型会遇到的问题
3. 切换后（正确修改）：适配 GPT-5.1 的正确写法

### Part 2: Responses API (新 API)
1. 切换前：GPT-4o 的 Responses API 调用
2. 切换后（不修改代码）：直接换模型会遇到的问题
3. 切换后（正确修改）：适配 GPT-5.1 的正确写法

## 环境准备

In [1]:
import requests
import json
from configparser import ConfigParser

config = ConfigParser()
config.read(r'c:\GitRepo\OpenAI-examples\.config')

# Azure OpenAI 配置
ENDPOINT_NAME = "jzdm-foundry-swn"  # Sweden Central 区域
API_KEY = config.get('AOAIEndpoints', ENDPOINT_NAME)
BASE_URL = f"https://{ENDPOINT_NAME}.openai.azure.com"

# 部署名称（请根据实际情况修改）
DEPLOYMENT_GPT4O = "gpt-4o-08-06-globalstandard"
DEPLOYMENT_GPT51 = "gpt-5.1-globalstandard"

print(f"Base URL: {BASE_URL}")
print(f"API Key: {API_KEY[:10]}...")
print(f"\nGPT-4o 部署: {DEPLOYMENT_GPT4O}")
print(f"GPT-5.1 部署: {DEPLOYMENT_GPT51}")

Base URL: https://jzdm-foundry-swn.openai.azure.com
API Key: 3U5X1Qewsm...

GPT-4o 部署: gpt-4o-08-06-globalstandard
GPT-5.1 部署: gpt-5.1-globalstandard


In [2]:
def print_response(response, title="Response"):
    """格式化打印 API 响应"""
    print("=" * 60)
    print(title)
    print("=" * 60)
    print(f"Status Code: {response.status_code}")
    
    try:
        data = response.json()
        print(f"\nResponse JSON:")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])  # 限制输出长度
        
        # 提取关键信息
        if response.status_code == 200:
            if "choices" in data:  # Chat Completions API
                print(f"\n--- 模型回复 ---")
                print(data["choices"][0]["message"]["content"][:500])
            elif "output_text" in data:  # Responses API
                print(f"\n--- 模型回复 ---")
                print(data["output_text"][:500])
    except:
        print(f"\nRaw Response: {response.text[:1000]}")

---
# Part 1: Chat Completions API

Chat Completions API 是最常用的 Azure OpenAI API，用于对话和文本生成。

**REST API 端点格式:**
```
POST https://{endpoint}/openai/deployments/{deployment-id}/chat/completions?api-version={api-version}
```

---
## 1️⃣ 切换前：GPT-4o 的标准调用

这是使用 GPT-4o (2024-08-06) 的标准 REST API 调用方式。

In [19]:
# ========================================
# ✅ GPT-4o Chat Completions API 标准调用
# ========================================

def call_chat_completions_gpt4o(prompt: str, max_tokens: int = 1000, temperature: float = 0.7):
    """
    GPT-4o 的标准 Chat Completions REST API 调用
    """
    url = f"{BASE_URL}/openai/deployments/{DEPLOYMENT_GPT4O}/chat/completions?api-version=2025-04-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "messages": [
            {"role": "system", "content": "你是一个有帮助的助手。"},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": max_tokens,        # ✅ GPT-4o 使用 max_tokens
        "temperature": temperature,       # ✅ GPT-4o 支持 temperature
        "top_p": 0.95,                    # ✅ GPT-4o 支持 top_p
        "frequency_penalty": 0,
        "presence_penalty": 0
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试 GPT-4o
print("测试 GPT-4o Chat Completions API")
response_4o = call_chat_completions_gpt4o("什么是量子计算？请用一句话简要说明。")
print_response(response_4o, "✅ GPT-4o Chat Completions 响应")

测试 GPT-4o Chat Completions API
✅ GPT-4o Chat Completions 响应
Status Code: 200

Response JSON:
{
  "choices": [
    {
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "protected_material_code": {
          "filtered": false,
          "detected": false
        },
        "protected_material_text": {
          "filtered": false,
          "detected": false
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        }
      },
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "annotations": [],
        "content": "量子计算是一种利用量子力学原理进行信息处理和计算的技术，能够在某些问题上实现比经典计算机更高效的计算能力。",
        "refusal": null,
        "role": "assistant"
      }
    }
  ],
  "cre

---
## 2️⃣ 切换后（不修改代码）：直接换模型的问题

如果只是把部署名称从 `gpt-4o` 换成 `gpt-5.1`，而不修改其他代码，**会遇到以下错误**：

### 会导致的错误：
1. **`max_tokens` 参数不被支持** - GPT-5.1 **不支持** `max_tokens`，必须使用 `max_completion_tokens`，否则会返回 400 错误
2. **`reasoning_effort` 默认为 `none`** - 不会启用推理能力
3. **如果设置 `reasoning_effort` 不为 `none`，同时使用 `temperature` 会报错**

In [20]:
# ========================================
# ❌ 问题演示 A：直接换模型名称（会报错！）
# ========================================

def call_chat_completions_gpt51_wrong_way_a(prompt: str, max_tokens: int = 1000, temperature: float = 0.7):
    """
    ❌ 错误方式 A：直接用 GPT-4o 的代码调用 GPT-5.1
    
    问题：
    1. 使用 max_tokens 而不是 max_completion_tokens（GPT-5.1 不支持，会直接报错！）
    2. 没有显式设置 reasoning_effort（默认为 none，无法利用推理能力）
    """
    url = f"{BASE_URL}/openai/deployments/{DEPLOYMENT_GPT51}/chat/completions?api-version=2025-04-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "messages": [
            {"role": "system", "content": "你是一个有帮助的助手。"},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": max_tokens,        # ❌ 旧参数，GPT-5.1 不支持，会报错！
        "temperature": temperature,       # ⚠️ 如果后面加了 reasoning_effort != none 会报错
        "top_p": 0.95,
        "frequency_penalty": 0,
        "presence_penalty": 0
        # ❌ 缺少 reasoning_effort 参数
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试
print("❌ 问题演示 A：直接换模型（不修改参数）- 预期会报错")
response_wrong_a = call_chat_completions_gpt51_wrong_way_a("什么是量子计算？请用一句话简要说明。")
print_response(response_wrong_a, "❌ GPT-5.1 直接替换（使用 max_tokens 会报错）")

print("\n" + "="*60)
print("❌ 错误原因说明：")
print("="*60)
print("1. GPT-5.1 不支持 max_tokens 参数，必须改用 max_completion_tokens")

❌ 问题演示 A：直接换模型（不修改参数）- 预期会报错
❌ GPT-5.1 直接替换（使用 max_tokens 会报错）
Status Code: 400

Response JSON:
{
  "error": {
    "message": "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.",
    "type": "invalid_request_error",
    "param": "max_tokens",
    "code": "unsupported_parameter"
  }
}

❌ 错误原因说明：
1. GPT-5.1 不支持 max_tokens 参数，必须改用 max_completion_tokens


In [21]:
# ========================================
# ❌ 问题演示 B：推理模式下使用 temperature（会报错！）
# ========================================

def call_chat_completions_gpt51_wrong_way_b(prompt: str):
    """
    ❌ 错误方式 B：在启用推理时使用 temperature 参数
    
    这会导致 BadRequestError 错误！
    """
    url = f"{BASE_URL}/openai/deployments/{DEPLOYMENT_GPT51}/chat/completions?api-version=2025-04-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "messages": [
            {"role": "system", "content": "你是一个有帮助的助手。"},
            {"role": "user", "content": prompt}
        ],
        "max_completion_tokens": 2000,
        "reasoning_effort": "medium",    # 启用推理
        "temperature": 0.7               # ❌ 推理模式下不能使用 temperature！
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试
print("❌ 问题演示 B：推理模式 + temperature（预期会报错）")
response_wrong_b = call_chat_completions_gpt51_wrong_way_b("1+1等于几？")
print_response(response_wrong_b, "❌ GPT-5.1 推理模式 + temperature 错误演示")

if response_wrong_b.status_code != 200:
    print("\n" + "="*60)
    print("📝 解决方案：")
    print("="*60)
    print("1. 如果需要推理能力：移除 temperature, top_p 等参数")
    print("2. 如果需要 temperature 控制：设置 reasoning_effort='none'")

❌ 问题演示 B：推理模式 + temperature（预期会报错）
❌ GPT-5.1 推理模式 + temperature 错误演示
Status Code: 400

Response JSON:
{
  "error": {
    "message": "Unsupported value: 'temperature' does not support 0.7 with this model. Only the default (1) value is supported.",
    "type": "invalid_request_error",
    "param": "temperature",
    "code": "unsupported_value"
  }
}

📝 解决方案：
1. 如果需要推理能力：移除 temperature, top_p 等参数
2. 如果需要 temperature 控制：设置 reasoning_effort='none'


---
## 3️⃣ 切换后（正确修改）：适配 GPT-5.1

### 修改要点：
1. ✅ 使用 `max_completion_tokens` 替代 `max_tokens`
2. ✅ 显式设置 `reasoning_effort` 参数
3. ✅ 如果启用推理，移除 `temperature`, `top_p` 等参数

In [22]:
# ========================================
# ✅ 正确方式 A：GPT-5.1 不启用推理（类似 GPT-4o 行为）
# ========================================

def call_chat_completions_gpt51_no_reasoning(prompt: str, max_completion_tokens: int = 1000, temperature: float = 0.7):
    """
    ✅ 正确方式：调用 GPT-5.1 但不使用推理能力
    
    适用场景：
    - 简单对话
    - 文本生成
    - 不需要复杂推理的任务
    - 需要保持与 GPT-4o 相似的行为
    """
    url = f"{BASE_URL}/openai/deployments/{DEPLOYMENT_GPT51}/chat/completions?api-version=2025-04-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "messages": [
            {"role": "system", "content": "你是一个有帮助的助手。"},
            {"role": "user", "content": prompt}
        ],
        "max_completion_tokens": max_completion_tokens,  # ✅ 新参数名
        "temperature": temperature,                       # ✅ reasoning_effort=none 时可用
        "reasoning_effort": "none"                        # ✅ 显式设置为 none
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试
print("✅ 正确方式 A：GPT-5.1 (无推理)")
response_correct_a = call_chat_completions_gpt51_no_reasoning("什么是量子计算？请用一句话简要说明。")
print_response(response_correct_a, "✅ GPT-5.1 Chat Completions (reasoning_effort=none)")

✅ 正确方式 A：GPT-5.1 (无推理)
✅ GPT-5.1 Chat Completions (reasoning_effort=none)
Status Code: 200

Response JSON:
{
  "choices": [
    {
      "content_filter_results": {},
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "annotations": [],
        "content": "量子计算是一种利用量子叠加和纠缠等量子力学特性来进行信息处理，从而在某些特定问题上有望比经典计算机更高效的计算方式。",
        "refusal": null,
        "role": "assistant"
      }
    }
  ],
  "created": 1768983097,
  "id": "chatcmpl-D0NbNJ00Be7B1IYHleIaYEoNdpoQ8",
  "model": "gpt-5.1-2025-11-13",
  "object": "chat.completion",
  "prompt_filter_results": [
    {
      "prompt_index": 0,
      "content_filter_results": {}
    }
  ],
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 58,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens": 32,
    "prompt_tokens_details": {
      "audio_tok

In [23]:
# ========================================
# ✅ 正确方式 B：GPT-5.1 启用推理能力
# ========================================

def call_chat_completions_gpt51_with_reasoning(prompt: str, max_completion_tokens: int = 4000, reasoning_effort: str = "medium"):
    """
    ✅ 正确方式：调用 GPT-5.1 并启用推理能力
    
    适用场景：
    - 复杂数学问题
    - 逻辑推理
    - 代码调试
    - 需要深度思考的任务
    
    reasoning_effort 选项：
    - none: 不推理（最快）
    - minimal: 最小推理
    - low: 低度推理
    - medium: 中度推理（推荐）
    - high: 高度推理
    """
    url = f"{BASE_URL}/openai/deployments/{DEPLOYMENT_GPT51}/chat/completions?api-version=2025-04-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "messages": [
            {"role": "system", "content": "你是一个擅长逻辑推理和数学的助手。请详细展示你的思考过程。"},
            {"role": "user", "content": prompt}
        ],
        "max_completion_tokens": max_completion_tokens,  # ✅ 推理需要更多 tokens
        "reasoning_effort": reasoning_effort             # ✅ 启用推理
        # ⚠️ 注意：启用推理时，不能使用 temperature, top_p 等参数！
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试推理能力
print("✅ 正确方式 B：GPT-5.1 (启用推理)")

math_problem = """请解决以下问题：

一个农夫有鸡和兔子。他数了数，一共有35个头，94只脚。
请问农夫有多少只鸡，多少只兔子？
"""

response_correct_b = call_chat_completions_gpt51_with_reasoning(math_problem, reasoning_effort="medium")
print_response(response_correct_b, "✅ GPT-5.1 Chat Completions (reasoning_effort=medium)")

✅ 正确方式 B：GPT-5.1 (启用推理)
✅ GPT-5.1 Chat Completions (reasoning_effort=medium)
Status Code: 200

Response JSON:
{
  "choices": [
    {
      "content_filter_results": {},
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "annotations": [],
        "content": "设：\n- 鸡有 \\(x\\) 只  \n- 兔子有 \\(y\\) 只  \n\n根据题意：\n\n1. 头的总数：  \n\\[\nx + y = 35\n\\]\n\n2. 脚的总数（鸡2只脚，兔4只脚）：  \n\\[\n2x + 4y = 94\n\\]\n\n将第二个方程两边同时除以2：  \n\\[\nx + 2y = 47\n\\]\n\n用这个方程减去头的方程：\n\\[\n(x + 2y) - (x + y) = 47 - 35\n\\]\n\\[\ny = 12\n\\]\n\n代入 \\(x + y = 35\\)：  \n\\[\nx + 12 = 35 \\Rightarrow x = 23\n\\]\n\n因此：\n- 鸡有 23 只  \n- 兔子有 12 只",
        "refusal": null,
        "role": "assistant"
      }
    }
  ],
  "created": 1768983109,
  "id": "chatcmpl-D0NbZu1OxY9hQPvnfyGFMPU9PLb1h",
  "model": "gpt-5.1-2025-11-13",
  "object": "chat.completion",
  "prompt_filter_results": [
    {
      "prompt_index": 0,
      "content_filter_results": {}
    }
  ],
  "system_fingerprint

---
# Part 2: Responses API (新 API)

Responses API 是 Azure OpenAI 的新 API，结合了 Chat Completions 和 Assistants API 的最佳特性。

**关键区别：**
- 使用 `v1` API 路径，不需要 `api-version` 参数
- 使用 `input` 而不是 `messages`
- 使用 `max_output_tokens` 而不是 `max_completion_tokens`
- 支持 `previous_response_id` 实现对话链

**REST API 端点格式:**
```
POST https://{endpoint}/openai/v1/responses
```

---
## 1️⃣ 切换前：GPT-4o 的 Responses API 调用

In [30]:
# ========================================
# ✅ GPT-4o Responses API 标准调用
# ========================================

def call_responses_api_gpt4o(prompt: str, max_output_tokens: int = 1000, temperature: float = 0.7):
    """
    GPT-4o 的标准 Responses API 调用
    
    注意：Responses API 使用 v1 路径，不需要 api-version
    """
    url = f"{BASE_URL}/openai/v1/responses"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "model": DEPLOYMENT_GPT4O,           # 使用部署名称
        "input": prompt,                      # Responses API 使用 input
        "max_output_tokens": max_output_tokens,  # ✅ Responses API 使用 max_output_tokens
        "temperature": temperature            # ✅ GPT-4o 支持 temperature
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试 GPT-4o Responses API
print("测试 GPT-4o Responses API")
response_4o_resp = call_responses_api_gpt4o("什么是机器学习？请用一句话简要说明。")
print_response(response_4o_resp, "✅ GPT-4o Responses API 响应")

测试 GPT-4o Responses API
✅ GPT-4o Responses API 响应
Status Code: 200

Response JSON:
{
  "id": "resp_0430f4832029a9e50069708eaee4188195bc7f15ca044e3675",
  "object": "response",
  "created_at": 1768984238,
  "status": "completed",
  "background": false,
  "content_filters": [
    {
      "blocked": false,
      "source_type": "prompt",
      "content_filter_raw": null,
      "content_filter_results": {
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "jailbreak": {
          "filtered": false,
          "detected": false
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        }
      },
      "content_filter_offsets": {
        "start_offset": 30,
        "end_offset": 48,
        "check_offset": 0
      }
    },
    {
   

---
## 2️⃣ 切换后（不修改代码）：直接换模型

### 原始代码不会报错

### 潜在问题：
1. **`reasoning_effort` 默认为 `none`** - 不会启用推理能力
2. **如果启用推理同时使用 `temperature` 会报错**

In [25]:
# ========================================
# ⚠️ 问题演示 A：Responses API 直接换模型
# ========================================

def call_responses_api_gpt51_wrong_way_a(prompt: str, max_output_tokens: int = 1000, temperature: float = 0.7):
    """
    ⚠️ 错误方式 A：直接用 GPT-4o 的代码调用 GPT-5.1
    
    问题：
    1. 没有显式设置 reasoning_effort（默认为 none，无法利用推理能力）
    """
    url = f"{BASE_URL}/openai/v1/responses"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "model": DEPLOYMENT_GPT51,
        "input": prompt,
        "max_output_tokens": max_output_tokens,
        "temperature": temperature
        # ❌ 缺少 reasoning_effort 参数
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试
print("⚠️ 问题演示 A：Responses API 直接换模型（不修改参数）")
response_resp_wrong_a = call_responses_api_gpt51_wrong_way_a("什么是机器学习？请用一句话简要说明。")
print_response(response_resp_wrong_a, "⚠️ GPT-5.1 Responses API 直接替换")

print("\n" + "="*60)
print("⚠️ 警告说明：")
print("="*60)
print("1. reasoning_effort 默认为 'none'，未启用推理能力")
print("2. 无法充分利用 GPT-5.1 的推理能力")

⚠️ 问题演示 A：Responses API 直接换模型（不修改参数）
⚠️ GPT-5.1 Responses API 直接替换
Status Code: 200

Response JSON:
{
  "id": "resp_0d2870080dae5d790069708a668d508197af27a4e006042488",
  "object": "response",
  "created_at": 1768983142,
  "status": "completed",
  "background": false,
  "content_filters": [
    {
      "blocked": false,
      "source_type": "prompt",
      "content_filter_raw": null,
      "content_filter_results": {},
      "content_filter_offsets": {
        "start_offset": 1429,
        "end_offset": 1447,
        "check_offset": 0
      }
    },
    {
      "blocked": false,
      "source_type": "completion",
      "content_filter_raw": null,
      "content_filter_results": {},
      "content_filter_offsets": {
        "start_offset": 0,
        "end_offset": 0,
        "check_offset": 0
      }
    }
  ],
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "max_output_tokens": 1000,
  "max_tool_calls": null,
  "model": "gpt-5.1-globalstandard",
  "output": [
 

In [26]:
# ========================================
# ❌ 问题演示 B：Responses API 推理模式下使用 temperature
# ========================================

def call_responses_api_gpt51_wrong_way_b(prompt: str):
    """
    ❌ 错误方式 B：在启用推理时使用 temperature 参数
    
    这会导致 BadRequestError 错误！
    """
    url = f"{BASE_URL}/openai/v1/responses"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "model": DEPLOYMENT_GPT51,
        "input": prompt,
        "max_output_tokens": 2000,
        "reasoning": {"effort": "medium"},  # ⚠️ Responses API 使用 reasoning.effort
        "temperature": 0.7                    # ❌ 推理模式下不能使用 temperature！
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试
print("❌ 问题演示 B：Responses API 推理模式 + temperature（预期会报错）")
response_resp_wrong_b = call_responses_api_gpt51_wrong_way_b("2+2等于几？")
print_response(response_resp_wrong_b, "❌ GPT-5.1 Responses API 推理 + temperature 错误演示")

if response_resp_wrong_b.status_code != 200:
    print("\n" + "="*60)
    print("📝 解决方案：")
    print("="*60)
    print("1. 如果需要推理能力：移除 temperature 参数")
    print("2. 如果需要 temperature 控制：不设置 reasoning 或设置为 none")

❌ 问题演示 B：Responses API 推理模式 + temperature（预期会报错）
❌ GPT-5.1 Responses API 推理 + temperature 错误演示
Status Code: 400

Response JSON:
{
  "error": {
    "message": "Unsupported parameter: 'temperature' is not supported with this model.",
    "type": "invalid_request_error",
    "param": "temperature",
    "code": null
  }
}

📝 解决方案：
1. 如果需要推理能力：移除 temperature 参数
2. 如果需要 temperature 控制：不设置 reasoning 或设置为 none


---
## 3️⃣ 切换后（正确修改）：适配 GPT-5.1

### Responses API 修改要点：
1. ✅ 使用 `reasoning` 对象设置推理级别（格式：`{"effort": "medium"}`）
2. ✅ 如果启用推理，移除 `temperature` 参数
3. ✅ 使用 `max_output_tokens` 控制输出长度

In [27]:
# ========================================
# ✅ 正确方式 A：Responses API GPT-5.1 不启用推理
# ========================================

def call_responses_api_gpt51_no_reasoning(prompt: str, max_output_tokens: int = 1000, temperature: float = 0.7):
    """
    ✅ 正确方式：Responses API 调用 GPT-5.1 不使用推理能力
    
    适用场景：
    - 简单对话
    - 文本生成
    - 需要保持与 GPT-4o 相似的行为
    """
    url = f"{BASE_URL}/openai/v1/responses"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "model": DEPLOYMENT_GPT51,
        "input": prompt,
        "max_output_tokens": max_output_tokens,
        "temperature": temperature,
        "reasoning": {"effort": "none"}  # ✅ 显式设置为 none
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试
print("✅ 正确方式 A：Responses API GPT-5.1 (无推理)")
response_resp_correct_a = call_responses_api_gpt51_no_reasoning("什么是机器学习？请用一句话简要说明。")
print_response(response_resp_correct_a, "✅ GPT-5.1 Responses API (reasoning.effort=none)")

✅ 正确方式 A：Responses API GPT-5.1 (无推理)
✅ GPT-5.1 Responses API (reasoning.effort=none)
Status Code: 200

Response JSON:
{
  "id": "resp_0f036d7b356815d10069708a7a2de48193b7692f1b64f72235",
  "object": "response",
  "created_at": 1768983162,
  "status": "completed",
  "background": false,
  "content_filters": [
    {
      "blocked": false,
      "source_type": "prompt",
      "content_filter_raw": null,
      "content_filter_results": {},
      "content_filter_offsets": {
        "start_offset": 1429,
        "end_offset": 1447,
        "check_offset": 0
      }
    },
    {
      "blocked": false,
      "source_type": "completion",
      "content_filter_raw": null,
      "content_filter_results": {},
      "content_filter_offsets": {
        "start_offset": 0,
        "end_offset": 0,
        "check_offset": 0
      }
    }
  ],
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "max_output_tokens": 1000,
  "max_tool_calls": null,
  "model": "gpt-5.1-globalstandard

In [28]:
# ========================================
# ✅ 正确方式 B：Responses API GPT-5.1 启用推理
# ========================================

def call_responses_api_gpt51_with_reasoning(prompt: str, max_output_tokens: int = 4000, reasoning_effort: str = "medium"):
    """
    ✅ 正确方式：Responses API 调用 GPT-5.1 启用推理能力
    
    适用场景：
    - 复杂数学问题
    - 逻辑推理
    - 代码调试
    
    reasoning.effort 选项：
    - none: 不推理
    - minimal: 最小推理
    - low: 低度推理
    - medium: 中度推理
    - high: 高度推理
    """
    url = f"{BASE_URL}/openai/v1/responses"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY
    }
    
    payload = {
        "model": DEPLOYMENT_GPT51,
        "input": prompt,
        "max_output_tokens": max_output_tokens,
        "reasoning": {"effort": reasoning_effort}  # ✅ 启用推理
        # ⚠️ 注意：启用推理时，不能使用 temperature！
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response

# 测试推理能力
print("✅ 正确方式 B：Responses API GPT-5.1 (启用推理)")

logic_problem = """请回答以下逻辑问题：

如果所有的猫都是动物，所有的动物都需要食物，那么所有的猫都需要食物吗？
请解释你的推理过程。
"""

response_resp_correct_b = call_responses_api_gpt51_with_reasoning(logic_problem, reasoning_effort="medium")
print_response(response_resp_correct_b, "✅ GPT-5.1 Responses API (reasoning.effort=medium)")

✅ 正确方式 B：Responses API GPT-5.1 (启用推理)
✅ GPT-5.1 Responses API (reasoning.effort=medium)
Status Code: 200

Response JSON:
{
  "id": "resp_0f30393e0a463f8a0069708a80b9848193aa5217852bd41115",
  "object": "response",
  "created_at": 1768983168,
  "status": "completed",
  "background": false,
  "content_filters": [
    {
      "blocked": false,
      "source_type": "prompt",
      "content_filter_raw": null,
      "content_filter_results": {},
      "content_filter_offsets": {
        "start_offset": 30,
        "end_offset": 91,
        "check_offset": 0
      }
    },
    {
      "blocked": false,
      "source_type": "completion",
      "content_filter_raw": null,
      "content_filter_results": {},
      "content_filter_offsets": {
        "start_offset": 0,
        "end_offset": 0,
        "check_offset": 0
      }
    }
  ],
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "max_output_tokens": 4000,
  "max_tool_calls": null,
  "model": "gpt-5.1-globalstandard"

---
# 📋 迁移检查清单

## Chat Completions API 迁移

| 修改项 | GPT-4o | GPT-5.1 |
|--------|--------|--------|
| 输出限制参数 | `max_tokens` | `max_completion_tokens` |
| 推理参数 | N/A | `reasoning_effort` |
| temperature | ✅ 始终可用 | ⚠️ 仅在 `reasoning_effort=none` 时可用 |

## Responses API 迁移

| 修改项 | GPT-4o | GPT-5.1 |
|--------|--------|--------|
| 端点 | `/openai/v1/responses` | `/openai/v1/responses` |
| 输出限制参数 | `max_output_tokens` | `max_output_tokens` |
| 推理参数 | N/A | `reasoning: {"effort": "..."} ` |
| temperature | ✅ 始终可用 | ⚠️ 仅在推理关闭时可用 |

---

## 🔗 参考资源

- [Azure OpenAI 模型退役文档](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/model-retirements)
- [Responses API 文档](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses)
- [推理模型指南](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/reasoning)
- [API 版本生命周期](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/api-version-lifecycle)